# Seminar Notebook 3.3: Scaling with Wordscores

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook covers the Wordscores method for supervised scaling of texts, replicating the example from the lecture using UK party manifestos.

## Set up steps

### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
if not os.path.exists(sdir):
    os.makedirs(sdir)

### Loading the UK Manifestos corpus

We will use a corpus of UK party manifestos from 1992 and 1997. The 1992 manifestos will serve as our labelled reference texts with known ideological positions, and we will use Wordscores to estimate positions for the 1997 manifestos.

In [ ]:
import requests

rfile = "https://raw.githubusercontent.com/lse-my459/data/master/UK_Manifestos.csv"
lfile = os.path.join(sdir, os.path.basename(rfile))

if not os.path.exists(lfile):
    r = requests.get(rfile)
    r.raise_for_status()
    with open(lfile, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {lfile}")
else:
    print(f"File already exists: {lfile}")

File already exists: /Users/r.hubert/LSE-MY459-WT26/SeminarWeek07/UK_Manifestos.csv


In [ ]:
import pandas as pd

df = pd.read_csv(lfile, dtype="object")
print(df)

    docname                                               text
0  Con_1992  Conservative Party 1992 \nThe Best Future for ...
1  Lab_1992  \nTHE LABOUR PARTY MANIFESTO\n\nTIME TO GET BR...
2   LD_1992  1992\nCHANGING BRITAIN FOR GOOD\n\nAFTER THE H...
3  Con_1997  THE CONSERVATIVE MANIFESTO 1997\n\nFOREWORD\nT...
4  Lab_1997  BRITAIN WILL BE BETTER WITH NEW LABOUR\n\n'OUR...
5   LD_1997  make the\ndifference\n\nThe Liberal Democrat M...


We have six documents: three from 1992 (Conservative, Labour, Liberal Democrat) and three from 1997. The 1992 documents are our labelled set, and the 1997 documents are our unlabelled set.

In [ ]:
# Indices for labelled (1992) and unlabelled (1997) documents
ltexts = [0, 1, 2]  # Con_1992, Lab_1992, LD_1992
utexts = [3, 4, 5]  # Con_1997, Lab_1997, LD_1997

### Preprocessing and creating the DFM

We preprocess the texts and create a document-feature matrix. For Wordscores, we do not need to remove stopwords or stem, but we do basic cleaning.

In [ ]:
import re
from collections import Counter
from sklearn.feature_extraction import DictVectorizer

# Tokenise and clean
df["tokens"] = df["text"].str.split(r"\s+")
df["tokens"] = df["tokens"].apply(lambda toks: [x.lower() for x in toks])
df["tokens"] = df["tokens"].apply(lambda toks: [re.sub(r"[^\w\-\']", "", x) for x in toks])
df["tokens"] = df["tokens"].apply(lambda toks: [x for x in toks if x != ""])

# Create term frequency counts
df["term_freqs"] = df["tokens"].map(Counter)

# Create DFM
dv = DictVectorizer()
dfm = dv.fit_transform(df["term_freqs"].to_list())
vocabulary = dv.get_feature_names_out()

print(dfm.shape)

(6, 7714)


### Trimming features

We trim the DFM to keep only features that appear at least once in the labelled documents. Words that don't appear in the labelled set cannot contribute to the word scores.

In [ ]:
# Calculate total term frequency in labelled documents
ttf_labelled = dfm[ltexts].sum(axis=0).A1

# Keep only features that appear in labelled documents
keep = ttf_labelled >= 1
dfm = dfm[:, keep]
vocabulary = vocabulary[keep]

print(dfm.shape)

(6, 5838)


## The Wordscores procedure

Wordscores estimates ideological positions for unlabelled texts by learning word-level scores from labelled reference texts. The procedure has several steps that we will work through one by one.

### Step 0: Setting up the labelled positions

We need to assign known ideological positions to our labelled texts. These come from expert codings on a left-right scale. Higher values indicate more right-wing positions.

### Step 1: Normalise the DFM

We create a relative document-feature matrix $\mathbf{F}$ by dividing each cell by its row sum. Each element $F_{ij}$ represents the proportion of document $i$'s words that are word $j$.

Let's look at a few features to see what $\mathbf{F}$ looks like.

### Step 2: Compute document probabilities for labelled texts

Next we compute a matrix $\mathbf{P}$ for the labelled documents only. Each element $P_{ij}$ is the probability that we are reading labelled document $i$ given that we see word $j$. We calculate this by dividing each column of $\mathbf{F}$ (for labelled docs) by its column sum.

Let's examine $\mathbf{P}$ for our sample words. Each column should sum to 1.

### Step 3: Calculate word scores

Now we calculate the word score for each word $j$. The word score $s_j$ is the weighted average of the labelled document positions, where the weights are the probabilities in $\mathbf{P}$:

$$s_j = \sum_{i \in L} \pi_i \times P_{ij}$$

Let's look at the word scores for our sample words.

We can also look at the most left-wing and right-wing words according to our word scores.

### Step 4: Calculate document scores

Finally, we calculate the estimated position for each document. We'll collect the results in a dataframe called `results`.

The document score $\hat{\pi}_i$ is the weighted average of word scores, where the weights are the word frequencies in $\mathbf{F}$:

$$\hat{\pi}_i = \sum_{j} F_{ij} \times s_j$$

Notice that the estimated scores are more "bunched" together than the original positions. This is because there are many non-discriminating words pulling the estimates toward the middle. We need to rescale.

### Step 5: Rescaling

We can rescale the scores using two common methods: the original LBG method and the Martin-Vanberg method. Both use affine transformations of the form $\hat{\pi}_i^{r} = a + b\hat{\pi}_i$. 

First, **LBG rescaling** transforms the raw scores so that the labelled documents have the same mean and standard deviation as the original positions.

Next, **Martin-Vanberg rescaling** anchors the transformation on the two most extreme labelled documents, ensuring they retain their original positions exactly.